# Evaluacion full-volume sagital final v1

Notebook academico para evaluar el modelo `sagittal_spider` final sobre un volumen 3D completo de un paciente del split de test final.

Alcance:

- Evaluacion volumetrica directa: procesa todas las slices sobre el eje sagital del checkpoint y reconstruye una segmentacion 3D.
- Integracion separada con el runtime existente: ejecuta `real_baseline` sobre una slice explicita del mismo volumen.
- No modifica endpoints, backend, frontend, runtime ni datasets.
- GCS se usa estrictamente en modo lectura. Las evidencias se guardan en Drive, nunca en GCS.

Advertencia academica: modelo asistivo de investigacion. Requiere revision profesional. No constituye diagnostico, no recomienda tratamientos, no reemplaza criterio clinico y no es un dispositivo medico validado. Un caso individual no demuestra generalizacion ni eficacia clinica.


## 2. Imports y dependencias


In [ ]:
from __future__ import annotations

import hashlib
import importlib.util
import json
import math
import os
import random
import re
import shutil
import subprocess
import sys
import tempfile
import time
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

REQUIRED_MODULES = {
    "google.cloud.storage": "google-cloud-storage",
    "google.auth": "google-auth",
    "pandas": "pandas",
    "numpy": "numpy",
    "PIL": "pillow",
    "SimpleITK": "SimpleITK",
    "matplotlib": "matplotlib",
    "torch": "torch",
}
missing = [pkg for module, pkg in REQUIRED_MODULES.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *sorted(set(missing))], check=True)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import SimpleITK as sitk
import torch
from google.cloud import storage
from PIL import Image

plt.ioff()

## 3. Configuracion inmutable


In [ ]:
def env_bool(name: str, default: bool) -> bool:
    value = os.getenv(name)
    if value is None:
        return default
    return value.strip().lower() in {"1", "true", "yes", "on"}


def env_int(name: str, default: int, low: int, high: int) -> int:
    value = int(os.getenv(name, str(default)))
    if value < low or value > high:
        raise ValueError(f"{name} fuera de rango: {value}")
    return value


@dataclass(frozen=True)
class FullVolumeEvaluationConfig:
    project_id: str = "pfi-asplanatti-fabrello-v1"
    bucket_name: str = "pfi-rm-lumbar-artifacts-2026-ef"
    release_prefix: str = "models/releases/sagittal_spider_final_v1"
    release_id: str = "sagittal_spider_final_v1"
    model_key: str = "sagittal_spider"
    version: str = "sagittal-spider-final-v1"
    expected_release_content_sha256: str = "7420ad4271fe634c970b2a543d1ef8fb1437888c99ca8bd5733a06e5f63e3e7e"
    expected_release_manifest_sha256: str = "d36d0c4fe183ba9a98f0a3471486be5dee1cf1fa820dc32b3a50177ce322be21"
    expected_model_sha256: str = "cf11dcc0ad77a7c787e64a796a2fd7398ef906add461cef4b3d61f1a5238e944"
    expected_training_commit: str = "6013e160f45c9263fd4ae50e864ceb37245323e2"
    expected_architecture_sha256: str = "d83f735cca9cbefc0e65dd8863466f4a528f205f3d674ebb73c49d68f8687c90"
    expected_dataset_manifest_sha256: str = "fa54c89a278d850021c0f91c0a27b3b5211c86301c9e4f125d75d517f39b793b"
    expected_training_index_sha256: str = "2720b7218c92870f6f0a000b57111ed36b5cf3b78c716f244f427ca7fee4a4ba"
    expected_patient_counts: dict[str, int] | None = None
    expected_slice_counts: dict[str, int] | None = None
    expected_classes: tuple[str, ...] = ("background", "vertebra_group", "canal", "disc_group")
    expected_num_classes: int = 4
    expected_base_channels: int = 16
    expected_target_size: tuple[int, int] = (256, 256)
    expected_sagittal_axis: int = 2
    source_training_output_dir: Path = Path("/content/drive/MyDrive/PFI_MVP/results/GCS_spider_final_training_v1")
    local_work_dir: Path = Path("/content/pfi_sagittal_final_full_volume_v1")
    local_release_dir: Path = Path("/content/pfi_sagittal_final_full_volume_v1/release")
    local_case_dir: Path = Path("/content/pfi_sagittal_final_full_volume_v1/case")
    drive_output_root: Path = Path("/content/drive/MyDrive/PFI_MVP/results/GCS_sagittal_final_full_volume_inference_v1")
    max_cases: int = env_int("PFI_FULL_VOLUME_MAX_CASES", 1, 1, 8)
    batch_size: int = env_int("PFI_FULL_VOLUME_BATCH_SIZE", 8, 1, 32)
    device: str = os.getenv("PFI_FULL_VOLUME_DEVICE", "auto")
    seed: int = 20260716
    gcs_write_allowed: bool = False
    human_review_required: bool = True
    not_clinical_diagnosis: bool = True

    def __post_init__(self) -> None:
        object.__setattr__(self, "expected_patient_counts", self.expected_patient_counts or {"train": 152, "val": 33, "test": 33})
        object.__setattr__(self, "expected_slice_counts", self.expected_slice_counts or {"train": 5271, "val": 1174, "test": 1218})


CFG = FullVolumeEvaluationConfig()
RUN_FULL_VOLUME_INFERENCE = env_bool("RUN_FULL_VOLUME_INFERENCE", False)
CONFIRM_FULL_VOLUME_INFERENCE = os.getenv("CONFIRM_FULL_VOLUME_INFERENCE", "")
EXPECTED_RUN_TOKEN = "RUN-SPIDER-FULL-VOLUME-7420ad42"
PFI_RELEASE_REPO_REF = os.getenv("PFI_RELEASE_REPO_REF", "main")
PFI_REPO_ROOT = Path(os.getenv("PFI_REPO_ROOT", "/content/PFI_MVPTest_Enzo_AImodule"))
REQUESTED_PATIENT_ID = os.getenv("PFI_FULL_VOLUME_TEST_PATIENT_ID", "").strip()
CLASS_IDS = tuple(range(CFG.expected_num_classes))
CLASS_NAMES = list(CFG.expected_classes)
gcs_read_operations = 0
gcs_write_operations = 0

assert CFG.release_prefix == "models/releases/sagittal_spider_final_v1"
assert CFG.model_key == "sagittal_spider"
assert CFG.version == "sagittal-spider-final-v1"
assert CFG.expected_num_classes == 4
assert CFG.expected_base_channels == 16
assert CFG.expected_target_size == (256, 256)
assert CFG.expected_sagittal_axis == 2
assert CFG.gcs_write_allowed is False
assert CFG.human_review_required is True
assert CFG.not_clinical_diagnosis is True

## 4. Montaje de Drive


In [ ]:
def running_in_colab() -> bool:
    try:
        import google.colab  # type: ignore
        return True
    except Exception:
        return False


def mount_drive_if_needed() -> None:
    if not running_in_colab():
        return
    drive_root = Path("/content/drive")
    my_drive = drive_root / "MyDrive"
    if my_drive.exists():
        return
    from google.colab import drive  # type: ignore
    if drive_root.exists() and any(drive_root.iterdir()):
        shutil.rmtree(drive_root)
    drive.mount(str(drive_root))
    if not my_drive.exists():
        raise RuntimeError("Drive no quedo montado.")


mount_drive_if_needed()
CFG.local_work_dir.mkdir(parents=True, exist_ok=True)
CFG.local_release_dir.mkdir(parents=True, exist_ok=True)
CFG.local_case_dir.mkdir(parents=True, exist_ok=True)

## 5. Preparacion o checkout del repositorio


In [ ]:
def run_git(args: list[str], cwd: Path | None = None, check: bool = True) -> subprocess.CompletedProcess:
    return subprocess.run(["git", *args], cwd=str(cwd) if cwd else None, text=True, capture_output=True, check=check)


def ensure_repo() -> Path:
    repo_url = "https://github.com/EnzoAA004/PFI_MVPTest_Enzo_AImodule.git"
    if PFI_REPO_ROOT.exists():
        probe = run_git(["rev-parse", "--show-toplevel"], PFI_REPO_ROOT, check=False)
        if probe.returncode != 0:
            raise RuntimeError("PFI_REPO_ROOT existe pero no es repo git.")
    else:
        PFI_REPO_ROOT.parent.mkdir(parents=True, exist_ok=True)
        run_git(["clone", repo_url, str(PFI_REPO_ROOT)])
    run_git(["fetch", "origin"], PFI_REPO_ROOT)
    run_git(["checkout", PFI_RELEASE_REPO_REF], PFI_REPO_ROOT)
    ai_service_path = PFI_REPO_ROOT / "ai_service"
    if str(ai_service_path) not in sys.path:
        sys.path.insert(0, str(ai_service_path))
    return PFI_REPO_ROOT


repo_root = ensure_repo()
from pfi_ai_service.model_architectures import build_checkpoint_model
from pfi_ai_service.model_manifest import validate_manifest

## 6. Autenticacion ADC de solo lectura


In [ ]:
def authenticate_read_only_client() -> storage.Client:
    if running_in_colab():
        from google.colab import auth  # type: ignore
        auth.authenticate_user()
    return storage.Client(project=CFG.project_id)


def count_gcs_read(action: str) -> None:
    global gcs_read_operations
    gcs_read_operations += 1


def assert_gcs_read_only_static() -> None:
    # Construye los nombres prohibidos sin dejarlos literales en el notebook versionado.
    forbidden = [
        "upload" + "_from_filename",
        "upload" + "_from_string",
        "blob." + "open(\"wb\")",
        "del" + "ete(",
        "rew" + "rite(",
        "comp" + "ose(",
        "make" + "_public",
        "GOOGLE" + "_APPLICATION" + "_CREDENTIALS",
        "private" + "_key",
    ]
    notebook_path = Path("notebooks/47_gcs_sagittal_final_full_volume_inference.ipynb")
    notebook_text = notebook_path.read_text(encoding="utf-8") if notebook_path.exists() else ""
    found = [token for token in forbidden if token in notebook_text]
    if found:
        raise RuntimeError(f"Operaciones no permitidas en notebook: {found}")


client = authenticate_read_only_client()
bucket = client.bucket(CFG.bucket_name)
bucket.reload()
count_gcs_read("bucket.reload")
if bucket.name != CFG.bucket_name:
    raise RuntimeError("Bucket inesperado.")
if getattr(bucket, "location", None) != "US-CENTRAL1":
    raise RuntimeError("Location de bucket inesperada.")
if getattr(bucket, "storage_class", None) != "STANDARD":
    raise RuntimeError("Storage class inesperado.")
assert gcs_write_operations == 0

## 7. Verificacion completa de la release GCS


In [ ]:
RELEASE_FILES = [
    "_SUCCESS.json",
    "publish_receipt.json",
    "release_manifest.json",
    "sagittal_spider_multiclass_final_best.pt",
    "sagittal_spider_multiclass_final_best.pt.manifest.json",
    "sagittal_spider_multiclass_final_best.pt.modelcard.md",
    "final_test_metrics.json",
    "training_history.csv",
    "training_history.json",
    "training_curves.png",
    "test_predictions_preview_informative.png",
    "gcs_spider_final_training_summary.json",
    "gcs_spider_final_training_evidence.json",
    "test_predictions_preview.png",
]


def sha256_stream(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as fh:
        for chunk in iter(lambda: fh.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def stable_json_dumps(payload: Any) -> str:
    return json.dumps(payload, indent=2, sort_keys=True, allow_nan=False, ensure_ascii=False)


def read_json(path: Path) -> dict[str, Any]:
    return json.loads(path.read_text(encoding="utf-8"))


def download_release_object(filename: str) -> Path:
    object_name = f"{CFG.release_prefix}/{filename}"
    blob = bucket.blob(object_name)
    if not blob.exists():
        count_gcs_read("blob.exists")
        raise FileNotFoundError(f"Objeto faltante en release: {filename}")
    count_gcs_read("blob.exists")
    blob.reload()
    count_gcs_read("blob.reload")
    target = CFG.local_release_dir / filename
    target.parent.mkdir(parents=True, exist_ok=True)
    blob.download_to_filename(str(target))
    count_gcs_read("blob.download_to_filename")
    if target.stat().st_size <= 0:
        raise RuntimeError(f"Objeto descargado vacio: {filename}")
    return target


release_paths = {filename: download_release_object(filename) for filename in RELEASE_FILES}
success = read_json(release_paths["_SUCCESS.json"])
receipt = read_json(release_paths["publish_receipt.json"])
release_manifest = read_json(release_paths["release_manifest.json"])
runtime_manifest = read_json(release_paths["sagittal_spider_multiclass_final_best.pt.manifest.json"])

if success.get("status") != "published_and_verified":
    raise RuntimeError("Release no publicada y verificada.")
if success.get("releaseId") != CFG.release_id:
    raise RuntimeError("releaseId inesperado.")
if success.get("releaseContentSha256") != CFG.expected_release_content_sha256:
    raise RuntimeError("releaseContentSha256 inesperado.")
if success.get("releaseManifestSha256") != CFG.expected_release_manifest_sha256:
    raise RuntimeError("releaseManifestSha256 inesperado.")
if success.get("publicationArtifactCount") != 12:
    raise RuntimeError("publicationArtifactCount inesperado.")
if success.get("remoteVerificationPassed") is not True:
    raise RuntimeError("remoteVerificationPassed no es true.")
if sha256_stream(release_paths["publish_receipt.json"]) != success.get("receiptSha256"):
    raise RuntimeError("SHA real de receipt no coincide con success.")
if receipt.get("releaseId") != CFG.release_id or receipt.get("releaseContentSha256") != CFG.expected_release_content_sha256:
    raise RuntimeError("Receipt no pertenece a esta release.")
if release_manifest.get("releaseId") != CFG.release_id:
    raise RuntimeError("Release manifest no pertenece a esta release.")
if sha256_stream(release_paths["release_manifest.json"]) != CFG.expected_release_manifest_sha256:
    raise RuntimeError("SHA real del release manifest inesperado.")
if any("sourcePath" in artifact for artifact in release_manifest.get("artifacts", [])):
    raise RuntimeError("El release manifest expone rutas locales.")
for artifact in release_manifest.get("artifacts", []):
    local_path = release_paths.get(artifact["filename"])
    if local_path is None:
        raise RuntimeError(f"Artifact no descargado: {artifact['filename']}")
    if local_path.stat().st_size != int(artifact["sizeBytes"]):
        raise RuntimeError(f"Size mismatch: {artifact['filename']}")
    if sha256_stream(local_path) != artifact["sha256"]:
        raise RuntimeError(f"SHA mismatch: {artifact['filename']}")
if sha256_stream(release_paths["sagittal_spider_multiclass_final_best.pt"]) != CFG.expected_model_sha256:
    raise RuntimeError("Model SHA inesperado.")
manifest_validation = validate_manifest(runtime_manifest, artifact_path=release_paths["sagittal_spider_multiclass_final_best.pt"])
if not manifest_validation.get("valid") or not manifest_validation.get("baselineReady") or manifest_validation.get("sha256Status") != "MATCH" or manifest_validation.get("validationErrors"):
    raise RuntimeError(f"Manifest runtime invalido: {manifest_validation}")

release_verification = {
    "releaseVerified": True,
    "releaseContentSha256": CFG.expected_release_content_sha256,
    "releaseManifestSha256": CFG.expected_release_manifest_sha256,
    "modelSha256": CFG.expected_model_sha256,
    "manifestValidation": manifest_validation,
    "gcsReadOperations": gcs_read_operations,
    "gcsWriteOperations": gcs_write_operations,
}

## 8. Descubrimiento y validacion del split final


In [ ]:
SMALL_ARTIFACT_PATTERNS = ("patient", "pair", "training_index", "slice_records", "split")


def file_sha256_if_exists(path: Path) -> str | None:
    return sha256_stream(path) if path.exists() and path.is_file() and path.stat().st_size > 0 else None


def canonical_csv_content_sha(path: Path) -> str:
    frame = pd.read_csv(path, dtype=str)
    text = frame.sort_index(axis=1).sort_values(list(frame.columns), kind="mergesort").to_csv(index=False)
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


def discover_split_artifacts(checkpoint: dict[str, Any]) -> dict[str, Path]:
    search_roots = [
        CFG.source_training_output_dir,
        CFG.source_training_output_dir.parent / "GCS_training_preflight",
        CFG.source_training_output_dir.parent / "GCS_spider_smoke_test",
    ]
    expected_hashes = {
        "patient_split": checkpoint.get("source_patient_split_sha256"),
        "pair_split_file": checkpoint.get("source_pair_split_file_sha256"),
        "pair_split_content": checkpoint.get("source_pair_split_content_sha256"),
        "slice_records_file": checkpoint.get("source_slice_records_file_sha256"),
        "slice_records_content": checkpoint.get("source_slice_records_content_sha256"),
        "training_index": checkpoint.get("source_training_index_sha256"),
    }
    candidates: list[Path] = []
    for root in search_roots:
        if root.exists():
            for path in root.rglob("*"):
                if path.is_file() and path.suffix.lower() in {".json", ".csv", ".parquet"}:
                    lowered = path.name.lower()
                    if any(pattern in lowered for pattern in SMALL_ARTIFACT_PATTERNS):
                        candidates.append(path)
    matches: dict[str, list[Path]] = {key: [] for key in expected_hashes}
    for path in candidates:
        raw_sha = file_sha256_if_exists(path)
        for key, expected in expected_hashes.items():
            if expected and raw_sha == expected:
                matches[key].append(path)
        if path.suffix.lower() == ".csv":
            try:
                content_sha = canonical_csv_content_sha(path)
            except Exception:
                content_sha = ""
            for key, expected in expected_hashes.items():
                if expected and content_sha == expected:
                    matches[key].append(path)
    selected: dict[str, Path] = {}

    for key, values in matches.items():
        unique = sorted(set(values))

        if len(unique) > 1:
            raise RuntimeError(
                f"Multiples artifacts incompatibles para {key}: "
                f"{[p.name for p in unique]}"
            )

        if unique:
            selected[key] = unique[0]

    # Para estos artifacts existen dos fingerprints posibles:
    # hash del archivo o hash canónico de su contenido.
    # Alcanza con que coincida uno de los dos.
    if not (
        selected.get("pair_split_file")
        or selected.get("pair_split_content")
    ):
        raise RuntimeError(
            "No se encontro un artifact canonico para pair_split "
            "por hash de archivo ni por hash de contenido."
        )

    if not (
        selected.get("slice_records_file")
        or selected.get("slice_records_content")
    ):
        raise RuntimeError(
            "No se encontro un artifact canonico para slice_records "
            "por hash de archivo ni por hash de contenido."
        )

    if "training_index" not in selected:
        raise RuntimeError(
            "No se encontro el training_index canonico."
        )


    return selected


checkpoint_preview = torch.load(release_paths["sagittal_spider_multiclass_final_best.pt"], map_location="cpu", weights_only=False)
split_artifacts = discover_split_artifacts(checkpoint_preview)
pair_split_path = split_artifacts.get("pair_split_file") or split_artifacts.get("pair_split_content")
slice_records_path = split_artifacts.get("slice_records_file") or split_artifacts.get("slice_records_content")
training_index_path = split_artifacts["training_index"]
pair_split_df = pd.read_csv(pair_split_path, dtype=str)
slice_records_df = pd.read_csv(slice_records_path, dtype=str)
training_index_df = pd.read_csv(training_index_path, dtype=str)

patient_counts = pair_split_df.groupby("split")["patient_id"].nunique().to_dict()
slice_counts = slice_records_df.groupby("split").size().to_dict()
if {key: int(value) for key, value in patient_counts.items()} != CFG.expected_patient_counts:
    raise RuntimeError(f"Conteos de pacientes inesperados: {patient_counts}")
if {key: int(value) for key, value in slice_counts.items()} != CFG.expected_slice_counts:
    raise RuntimeError(f"Conteos de slices inesperados: {slice_counts}")
split_sets = {split: set(pair_split_df.loc[pair_split_df["split"] == split, "patient_id"].astype(str)) for split in ["train", "val", "test"]}
if any(split_sets[a] & split_sets[b] for a in split_sets for b in split_sets if a < b):
    raise RuntimeError("Leakage de pacientes entre splits.")
if len(set.union(*split_sets.values())) != 218:
    raise RuntimeError("Union total de pacientes inesperada.")
if checkpoint_preview.get("source_manifest_sha256") != CFG.expected_dataset_manifest_sha256:
    raise RuntimeError("Checkpoint con manifest SHA inesperado.")
if checkpoint_preview.get("source_training_index_sha256") != CFG.expected_training_index_sha256:
    raise RuntimeError("Checkpoint con training index SHA inesperado.")

split_verification = {
    "splitVerified": True,
    "patientCounts": {k: int(v) for k, v in patient_counts.items()},
    "sliceCounts": {k: int(v) for k, v in slice_counts.items()},
    "patientSetsDisjoint": True,
    "totalPatients": 218,
    "datasetManifestSha256": CFG.expected_dataset_manifest_sha256,
    "trainingIndexSha256": CFG.expected_training_index_sha256,
    "artifacts": {k: v.name for k, v in split_artifacts.items()},
}

## 9. Seleccion deterministica del caso de test


In [ ]:
def short_hash(text: str, length: int = 16) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()[:length]


def object_from_gs_uri(uri: str) -> str:
    prefix = f"gs://{CFG.bucket_name}/"
    if not str(uri).startswith(prefix):
        raise ValueError(f"URI fuera del bucket esperado: {uri}")
    return str(uri)[len(prefix):]


test_pairs = pair_split_df.loc[pair_split_df["split"] == "test"].copy()
test_pairs["patient_id"] = test_pairs["patient_id"].astype(str)
if REQUESTED_PATIENT_ID:
    if REQUESTED_PATIENT_ID not in split_sets["test"]:
        raise RuntimeError("El override de paciente no pertenece al split test.")
    if REQUESTED_PATIENT_ID in split_sets["train"] or REQUESTED_PATIENT_ID in split_sets["val"]:
        raise RuntimeError("El override de paciente aparece en train/validation.")
    selected_pair_rows = test_pairs.loc[test_pairs["patient_id"] == REQUESTED_PATIENT_ID].copy()
    selection_mode = "explicit_test_patient_override"
else:
    merged_pairs = test_pairs.merge(training_index_df, on="pair_id", how="left", validate="one_to_one")
    for col in ["image_gcs_uri", "mask_gcs_uri"]:
        if merged_pairs[col].isna().any():
            raise RuntimeError(f"training index sin {col} para algun pair de test.")
    merged_pairs["selection_hash"] = merged_pairs.apply(
        lambda row: short_hash(f"{CFG.seed}|{row.patient_id}|{object_from_gs_uri(row.image_gcs_uri)}|{object_from_gs_uri(row.mask_gcs_uri)}", 64),
        axis=1,
    )
    selected_pair_rows = merged_pairs.sort_values("selection_hash", kind="mergesort").head(1).copy()
    selection_mode = "deterministic_test_hash_order"

selected_pair = selected_pair_rows.iloc[0].to_dict()
if "image_gcs_uri" not in selected_pair:
    selected_pair = selected_pair_rows.merge(training_index_df, on="pair_id", how="left", validate="one_to_one").iloc[0].to_dict()
selected_patient_id = str(selected_pair["patient_id"])
patient_hash = short_hash(f"{CFG.seed}|{selected_patient_id}", 16)
case_alias = f"SPIDER-TEST-{patient_hash[:8]}"
image_object = object_from_gs_uri(selected_pair["image_gcs_uri"])
mask_object = object_from_gs_uri(selected_pair["mask_gcs_uri"])
selected_case_manifest = {
    "caseAlias": case_alias,
    "patientIdHash": patient_hash,
    "split": "test",
    "selectionMode": selection_mode,
    "imageObject": image_object,
    "maskObject": mask_object,
    "imageSizeBytes": int(selected_pair.get("image_size_bytes", 0)),
    "maskSizeBytes": int(selected_pair.get("mask_size_bytes", 0)),
    "datasetManifestSha256": CFG.expected_dataset_manifest_sha256,
    "trainingIndexSha256": CFG.expected_training_index_sha256,
    "releaseContentSha256": CFG.expected_release_content_sha256,
}

## 10. Descarga temporal del volumen y ground truth


In [ ]:
def supported_medical_suffix(path: Path) -> bool:
    name = path.name.lower()
    return name.endswith((".mha", ".mhd", ".nii", ".nii.gz", ".npy"))


def download_case_object(object_name: str, target_name: str) -> Path:
    blob = bucket.blob(object_name)
    if not blob.exists():
        count_gcs_read("blob.exists")
        raise FileNotFoundError(f"Objeto de caso faltante: {target_name}")
    count_gcs_read("blob.exists")
    blob.reload()
    count_gcs_read("blob.reload")
    target = CFG.local_case_dir / target_name
    blob.download_to_filename(str(target))
    count_gcs_read("blob.download_to_filename")
    if target.stat().st_size <= 0:
        raise RuntimeError(f"Descarga vacia: {target_name}")
    return target


image_suffix = Path(image_object).suffix or ".mha"
mask_suffix = Path(mask_object).suffix or ".mha"
input_volume_path = download_case_object(image_object, f"input_volume{image_suffix}")
ground_truth_path = download_case_object(mask_object, f"ground_truth_volume{mask_suffix}")
if not supported_medical_suffix(input_volume_path) or not supported_medical_suffix(ground_truth_path):
    raise RuntimeError("Extension medica no soportada para volumen o GT.")
selected_case_manifest["imageLocalSha256"] = sha256_stream(input_volume_path)
selected_case_manifest["maskLocalSha256"] = sha256_stream(ground_truth_path)

## 11. Validacion geometrica y de etiquetas


In [ ]:
def read_sitk_array(path: Path) -> tuple[sitk.Image, np.ndarray]:
    image = sitk.ReadImage(str(path))
    array = sitk.GetArrayFromImage(image)
    return image, np.asarray(array)


def geometry_payload(image: sitk.Image) -> dict[str, Any]:
    return {
        "shape": list(reversed(image.GetSize())),
        "spacing": [float(v) for v in image.GetSpacing()],
        "origin": [float(v) for v in image.GetOrigin()],
        "direction": [float(v) for v in image.GetDirection()],
    }


def assert_same_geometry(image: sitk.Image, mask: sitk.Image) -> None:
    if image.GetSize() != mask.GetSize():
        raise RuntimeError("Image/mask size incompatible.")
    for name, getter in [("spacing", "GetSpacing"), ("origin", "GetOrigin"), ("direction", "GetDirection")]:
        left = np.asarray(getattr(image, getter)(), dtype=float)
        right = np.asarray(getattr(mask, getter)(), dtype=float)
        if not np.allclose(left, right, rtol=0.0, atol=1e-6):
            raise RuntimeError(f"Image/mask {name} incompatible.")


def normalize_label_mapping(raw_mapping: dict[str, Any]) -> dict[int, int]:
    mapping: dict[int, int] = {}
    for key, value in raw_mapping.items():
        if isinstance(value, list):
            class_id = int(key)
            for raw_label in value:
                mapping[int(raw_label)] = class_id
        else:
            mapping[int(key)] = int(value)
    if set(mapping.values()) != set(CLASS_IDS):
        raise RuntimeError("label_group_mapping no produce exactamente clases 0..3.")
    return dict(sorted(mapping.items()))


def apply_label_map(mask: np.ndarray, label_map: dict[int, int]) -> np.ndarray:
    raw_labels = sorted(int(v) for v in np.unique(mask))
    unknown = [label for label in raw_labels if label not in label_map]
    if unknown:
        raise RuntimeError(f"Labels crudos sin mapping: {unknown[:20]}")
    grouped = np.zeros(mask.shape, dtype=np.int64)
    for raw_label, class_id in label_map.items():
        grouped[np.asarray(mask) == raw_label] = class_id
    if not set(int(v) for v in np.unique(grouped)).issubset(set(CLASS_IDS)):
        raise RuntimeError("Mapping produjo labels fuera de 0..3.")
    return grouped


image_sitk, image_volume = read_sitk_array(input_volume_path)
mask_sitk, raw_mask_volume = read_sitk_array(ground_truth_path)
assert_same_geometry(image_sitk, mask_sitk)
if image_volume.ndim != 3 or raw_mask_volume.ndim != 3 or image_volume.shape != raw_mask_volume.shape:
    raise RuntimeError("Se esperan volumen y mascara 3D con shape identica.")
if not np.isfinite(image_volume).all() or float(np.nanmax(image_volume)) == float(np.nanmin(image_volume)):
    raise RuntimeError("Imagen invalida: no finita o constante.")
label_map = normalize_label_mapping(checkpoint_preview["label_group_mapping"])
grouped_mask_volume = apply_label_map(raw_mask_volume, label_map)
label_profile = {
    "rawLabels": sorted(int(v) for v in np.unique(raw_mask_volume)),
    "groupedLabels": sorted(int(v) for v in np.unique(grouped_mask_volume)),
    "voxelsByClass": {CLASS_NAMES[i]: int(np.count_nonzero(grouped_mask_volume == i)) for i in CLASS_IDS},
    "slicesWithForeground": int(sum(np.any(np.take(grouped_mask_volume, idx, axis=CFG.expected_sagittal_axis) > 0) for idx in range(grouped_mask_volume.shape[CFG.expected_sagittal_axis]))),
    "absentClasses": [CLASS_NAMES[i] for i in CLASS_IDS if not np.any(grouped_mask_volume == i)],
}

## 12. Carga estricta del checkpoint


In [ ]:
checkpoint = torch.load(release_paths["sagittal_spider_multiclass_final_best.pt"], map_location="cpu", weights_only=False)
model, runtime_meta = build_checkpoint_model(CFG.model_key, checkpoint)
model.eval()
if model.training:
    raise RuntimeError("Modelo no quedo en eval mode.")
if int(runtime_meta["numClasses"]) != CFG.expected_num_classes:
    raise RuntimeError("numClasses inesperado.")
if int(runtime_meta["baseChannels"]) != CFG.expected_base_channels:
    raise RuntimeError("baseChannels inesperado.")
if tuple(runtime_meta["targetSize"]) != CFG.expected_target_size:
    raise RuntimeError("targetSize inesperado.")
if int(checkpoint.get("sagittal_axis")) != CFG.expected_sagittal_axis:
    raise RuntimeError("sagittal_axis inesperado.")
if list(checkpoint.get("classes")) != CLASS_NAMES:
    raise RuntimeError("Clases incompatibles.")
if checkpoint.get("source_repository_commit") != CFG.expected_training_commit:
    raise RuntimeError("source_repository_commit inesperado.")
if checkpoint.get("source_model_architecture_sha256") != CFG.expected_architecture_sha256:
    raise RuntimeError("source_model_architecture_sha256 inesperado.")
if checkpoint.get("source_manifest_sha256") != CFG.expected_dataset_manifest_sha256:
    raise RuntimeError("source_manifest_sha256 inesperado.")
if checkpoint.get("source_training_index_sha256") != CFG.expected_training_index_sha256:
    raise RuntimeError("source_training_index_sha256 inesperado.")
with torch.inference_mode():
    smoke_logits = model(torch.zeros(1, 1, 256, 256))
if list(smoke_logits.shape) != [1, 4, 256, 256] or not torch.isfinite(smoke_logits).all().item():
    raise RuntimeError("Smoke tensor produjo salida invalida.")
strict_load_success = True

## 13. Validacion del preprocesamiento


In [ ]:
def canonicalize_spider_array_with_layout(
    arr: np.ndarray,
) -> tuple[np.ndarray, bool]:
    arr = np.asarray(arr)

    moved_axis0_to_last = bool(
        arr.ndim == 3
        and arr.shape[0] <= 64
        and arr.shape[-1] > 64
    )

    if moved_axis0_to_last:
        return np.moveaxis(arr, 0, -1), True

    return arr, False


# Conservar la orientación original para reconstrucción posterior.
# Los guards permiten volver a ejecutar esta celda sin transformar dos veces.
if "image_volume_native" not in globals():
    image_volume_native = np.asarray(image_volume)

if "grouped_mask_volume_native" not in globals():
    grouped_mask_volume_native = np.asarray(grouped_mask_volume)


image_volume, image_axis0_moved_to_last = (
    canonicalize_spider_array_with_layout(
        image_volume_native
    )
)

grouped_mask_volume, mask_axis0_moved_to_last = (
    canonicalize_spider_array_with_layout(
        grouped_mask_volume_native
    )
)


if image_axis0_moved_to_last != mask_axis0_moved_to_last:
    raise RuntimeError(
        "Imagen y máscara necesitaron transformaciones "
        "de orientación diferentes."
    )

if image_volume.shape != grouped_mask_volume.shape:
    raise RuntimeError(
        "Imagen y máscara canonicalizadas tienen shapes diferentes: "
        f"{image_volume.shape} != {grouped_mask_volume.shape}"
    )


def robust_normalize(image: np.ndarray) -> np.ndarray:
    arr = np.asarray(image, dtype=np.float32)
    arr = np.nan_to_num(
        arr,
        nan=0.0,
        posinf=0.0,
        neginf=0.0,
    )

    p1, p99 = np.percentile(arr, [1, 99])

    if p99 <= p1:
        return np.zeros(
            arr.shape,
            dtype=np.float32,
        )

    arr = (arr - p1) / (p99 - p1)

    return np.clip(
        arr,
        0.0,
        1.0,
    ).astype(np.float32)


def resize_image(
    image: np.ndarray,
    size: tuple[int, int],
) -> np.ndarray:
    pil = Image.fromarray(
        (
            np.clip(image, 0.0, 1.0)
            * 255
        ).astype(np.uint8)
    )

    resized = pil.resize(
        (size[1], size[0]),
        Image.Resampling.BILINEAR,
    )

    return (
        np.asarray(resized).astype(np.float32)
        / 255.0
    ).astype(np.float32)


def resize_mask(
    mask: np.ndarray,
    size: tuple[int, int],
) -> np.ndarray:
    pil = Image.fromarray(
        mask.astype(np.int32)
    )

    resized = pil.resize(
        (size[1], size[0]),
        Image.Resampling.NEAREST,
    )

    out = np.asarray(
        resized,
        dtype=np.int64,
    )

    observed_labels = {
        int(value)
        for value in np.unique(out)
    }

    if not observed_labels.issubset(set(CLASS_IDS)):
        raise RuntimeError(
            "Resize nearest produjo labels fuera de 0..3."
        )

    return out


def extract_slice(
    arr: np.ndarray,
    slice_index: int,
    slice_axis: int,
) -> np.ndarray:
    arr = np.asarray(arr)

    if arr.ndim == 2:
        return arr

    if arr.ndim != 3:
        raise RuntimeError(
            f"Se esperaba array 2D o 3D, recibido: {arr.shape}"
        )

    slice_axis = int(slice_axis)
    slice_index = int(slice_index)
    slice_count = int(arr.shape[slice_axis])

    if slice_index < 0 or slice_index >= slice_count:
        raise IndexError(
            f"slice_index={slice_index} fuera de rango "
            f"para axis={slice_axis}, count={slice_count}, "
            f"shape={arr.shape}"
        )

    return np.take(
        arr,
        slice_index,
        axis=slice_axis,
    )


def preprocess_pair(
    slice_index: int,
) -> tuple[np.ndarray, np.ndarray]:
    image_slice = extract_slice(
        image_volume,
        slice_index,
        CFG.expected_sagittal_axis,
    )

    mask_slice = extract_slice(
        grouped_mask_volume,
        slice_index,
        CFG.expected_sagittal_axis,
    )

    if image_slice.shape != mask_slice.shape:
        raise RuntimeError(
            "Slice de imagen y máscara con shapes diferentes: "
            f"{image_slice.shape} != {mask_slice.shape}"
        )

    image_model = resize_image(
        robust_normalize(image_slice),
        CFG.expected_target_size,
    )

    mask_model = resize_mask(
        mask_slice,
        CFG.expected_target_size,
    )

    if (
        image_model.shape != CFG.expected_target_size
        or mask_model.shape != CFG.expected_target_size
    ):
        raise RuntimeError(
            "Shape model-space inesperado."
        )

    if (
        image_model.dtype != np.float32
        or not np.isfinite(image_model).all()
    ):
        raise RuntimeError(
            "Imagen preprocesada inválida."
        )

    mask_labels = {
        int(value)
        for value in np.unique(mask_model)
    }

    if not mask_labels.issubset(set(CLASS_IDS)):
        raise RuntimeError(
            "GT model-space contiene labels inválidos."
        )

    if (
        float(image_model.min()) < 0.0
        or float(image_model.max()) > 1.0
    ):
        raise RuntimeError(
            "Normalización fuera de rango."
        )

    return image_model, mask_model


sagittal_axis = int(
    CFG.expected_sagittal_axis
)

image_slice_count = int(
    image_volume.shape[sagittal_axis]
)

mask_slice_count = int(
    grouped_mask_volume.shape[sagittal_axis]
)

if image_slice_count != mask_slice_count:
    raise RuntimeError(
        "Imagen y máscara tienen diferente cantidad "
        "de slices sagitales."
    )

foreground_by_slice = [
    int(
        np.count_nonzero(
            np.take(
                grouped_mask_volume,
                index,
                axis=sagittal_axis,
            ) > 0
        )
    )
    for index in range(mask_slice_count)
]

integration_slice_index = int(
    np.argmax(foreground_by_slice)
)

if not (
    0
    <= integration_slice_index
    < image_slice_count
):
    raise RuntimeError(
        "Índice de integración fuera de rango "
        "después de canonicalizar."
    )

sample_image_model, sample_mask_model = (
    preprocess_pair(
        integration_slice_index
    )
)

preprocessing_validation = {
    "algorithm": (
        "robust_percentile_1_99_"
        "uint8_bilinear_image_"
        "nearest_mask"
    ),
    "originalImageShape": list(
        image_volume_native.shape
    ),
    "canonicalImageShape": list(
        image_volume.shape
    ),
    "originalMaskShape": list(
        grouped_mask_volume_native.shape
    ),
    "canonicalMaskShape": list(
        grouped_mask_volume.shape
    ),
    "axis0MovedToLast": bool(
        image_axis0_moved_to_last
    ),
    "sagittalAxis": sagittal_axis,
    "sliceCount": image_slice_count,
    "targetSize": list(
        CFG.expected_target_size
    ),
    "sampleSlice": integration_slice_index,
    "sampleForegroundVoxels": int(
        foreground_by_slice[
            integration_slice_index
        ]
    ),
    "imageShape": list(
        sample_image_model.shape
    ),
    "maskShape": list(
        sample_mask_model.shape
    ),
    "imageFinite": bool(
        np.isfinite(
            sample_image_model
        ).all()
    ),
    "maskLabels": sorted(
        int(value)
        for value in np.unique(
            sample_mask_model
        )
    ),
}

print(
    stable_json_dumps(
        preprocessing_validation
    )
)

## 14. Gate de ejecucion volumetrica


In [ ]:
run_gate_enabled = RUN_FULL_VOLUME_INFERENCE and CONFIRM_FULL_VOLUME_INFERENCE == EXPECTED_RUN_TOKEN
if RUN_FULL_VOLUME_INFERENCE and CONFIRM_FULL_VOLUME_INFERENCE != EXPECTED_RUN_TOKEN:
    raise RuntimeError("RUN_FULL_VOLUME_INFERENCE activo sin token correcto.")
plan_summary = {
    "executionStatus": "preflight_ready" if not run_gate_enabled else "full_volume_execution_enabled",
    "runFullVolumeInference": bool(run_gate_enabled),
    "volumeShape": list(image_volume.shape),
    "sagittalAxis": CFG.expected_sagittal_axis,
    "sliceCount": int(image_volume.shape[CFG.expected_sagittal_axis]),
    "batchSize": CFG.batch_size,
    "gcsWriteOperations": gcs_write_operations,
}
if not run_gate_enabled:
    print(stable_json_dumps(plan_summary))

## 15. Inferencia de todas las slices


In [ ]:
def resolve_device() -> torch.device:
    requested = CFG.device.lower().strip()
    if requested not in {"auto", "cpu", "cuda"}:
        raise ValueError("PFI_FULL_VOLUME_DEVICE debe ser auto, cpu o cuda.")
    if requested == "cuda" and torch.cuda.is_available():
        return torch.device("cuda")
    if requested == "cuda":
        raise RuntimeError("CUDA solicitada pero no disponible.")
    return torch.device("cuda" if requested == "auto" and torch.cuda.is_available() else "cpu")


def predict_all_slices() -> dict[str, Any]:
    device = resolve_device()
    model.to(device)
    slice_count = int(image_volume.shape[CFG.expected_sagittal_axis])
    pred_model = np.zeros((slice_count, *CFG.expected_target_size), dtype=np.uint8)
    gt_model = np.zeros((slice_count, *CFG.expected_target_size), dtype=np.uint8)
    conf_model = np.zeros((slice_count, *CFG.expected_target_size), dtype=np.float32)
    started = time.time()
    processed = 0
    with torch.inference_mode():
        for start in range(0, slice_count, CFG.batch_size):
            stop = min(start + CFG.batch_size, slice_count)
            images = []
            masks = []
            for idx in range(start, stop):
                img, gt = preprocess_pair(idx)
                images.append(img)
                masks.append(gt)
            tensor = torch.from_numpy(np.stack(images)[:, None]).float().to(device)
            logits = model(tensor)
            probs = torch.softmax(logits, dim=1)
            pred = torch.argmax(probs, dim=1).detach().cpu().numpy().astype(np.uint8)
            conf = torch.max(probs, dim=1).values.detach().cpu().numpy().astype(np.float32)
            pred_model[start:stop] = pred
            gt_model[start:stop] = np.stack(masks).astype(np.uint8)
            conf_model[start:stop] = conf
            processed += stop - start
            del tensor, logits, probs
            if device.type == "cuda":
                torch.cuda.empty_cache()
    elapsed = max(time.time() - started, 1e-9)
    if processed != slice_count:
        raise RuntimeError("No se procesaron todas las slices exactamente una vez.")
    return {
        "predictionModelSpace": pred_model,
        "groundTruthModelSpace": gt_model,
        "confidenceModelSpace": conf_model,
        "stats": {
            "totalSlices": slice_count,
            "batches": int(math.ceil(slice_count / CFG.batch_size)),
            "elapsedSeconds": elapsed,
            "secondsPerSlice": elapsed / slice_count,
            "device": str(device),
            "cudaMaxMemoryAllocated": int(torch.cuda.max_memory_allocated()) if device.type == "cuda" else None,
            "predictedLabels": sorted(int(v) for v in np.unique(pred_model)),
            "emptyPredictionSlices": int(sum(not np.any(pred_model[i] > 0) for i in range(slice_count))),
            "foregroundPredictionSlices": int(sum(np.any(pred_model[i] > 0) for i in range(slice_count))),
        },
    }


volume_prediction = predict_all_slices() if run_gate_enabled else None

## 16. Reconstruccion 3D


In [ ]:
def resize_prediction_to_native(
    mask: np.ndarray,
    native_shape: tuple[int, int],
) -> np.ndarray:
    pil = Image.fromarray(mask.astype(np.uint8))

    resized = pil.resize(
        (native_shape[1], native_shape[0]),
        Image.Resampling.NEAREST,
    )

    return np.asarray(
        resized,
        dtype=np.uint8,
    )


def restore_sitk_array_order(
    canonical_array: np.ndarray,
) -> np.ndarray:
    canonical_array = np.asarray(
        canonical_array
    )

    if image_axis0_moved_to_last:
        restored = np.moveaxis(
            canonical_array,
            -1,
            0,
        )
    else:
        restored = canonical_array

    expected_shape = tuple(
        int(value)
        for value in image_volume_native.shape
    )

    if restored.shape != expected_shape:
        raise RuntimeError(
            "La predicción restaurada no coincide "
            "con el array nativo de SimpleITK: "
            f"{restored.shape} != {expected_shape}"
        )

    return restored


def reconstruct_native(
    pred_model: np.ndarray,
) -> tuple[np.ndarray, sitk.Image]:
    canonical_shape = tuple(
        int(value)
        for value in image_volume.shape
    )

    canonical_pred = np.zeros(
        canonical_shape,
        dtype=np.uint8,
    )

    slice_count = int(
        canonical_shape[
            CFG.expected_sagittal_axis
        ]
    )

    if pred_model.shape[0] != slice_count:
        raise RuntimeError(
            "Cantidad de predicciones incompatible "
            "con el volumen canonicalizado."
        )

    for idx in range(slice_count):
        native_slice_shape = np.take(
            image_volume,
            idx,
            axis=CFG.expected_sagittal_axis,
        ).shape

        restored_slice = (
            resize_prediction_to_native(
                pred_model[idx],
                native_slice_shape,
            )
        )

        if CFG.expected_sagittal_axis == 0:
            canonical_pred[idx, :, :] = (
                restored_slice
            )
        elif CFG.expected_sagittal_axis == 1:
            canonical_pred[:, idx, :] = (
                restored_slice
            )
        elif CFG.expected_sagittal_axis == 2:
            canonical_pred[:, :, idx] = (
                restored_slice
            )
        else:
            raise RuntimeError(
                "Eje sagital fuera de rango."
            )

    sitk_array_pred = restore_sitk_array_order(
        canonical_pred
    )

    pred_sitk = sitk.GetImageFromArray(
        sitk_array_pred
    )

    if pred_sitk.GetSize() != image_sitk.GetSize():
        raise RuntimeError(
            "La imagen reconstruida no conserva "
            "el tamaño físico original: "
            f"{pred_sitk.GetSize()} != "
            f"{image_sitk.GetSize()}"
        )

    pred_sitk.CopyInformation(image_sitk)

    if not np.allclose(
        pred_sitk.GetSpacing(),
        image_sitk.GetSpacing(),
        rtol=0.0,
        atol=1e-8,
    ):
        raise RuntimeError(
            "No se preservó spacing."
        )

    if not np.allclose(
        pred_sitk.GetOrigin(),
        image_sitk.GetOrigin(),
        rtol=0.0,
        atol=1e-8,
    ):
        raise RuntimeError(
            "No se preservó origin."
        )

    if not np.allclose(
        pred_sitk.GetDirection(),
        image_sitk.GetDirection(),
        rtol=0.0,
        atol=1e-8,
    ):
        raise RuntimeError(
            "No se preservó direction."
        )

    return sitk_array_pred, pred_sitk


if run_gate_enabled:
    (
        predicted_native,
        predicted_native_sitk,
    ) = reconstruct_native(
        volume_prediction[
            "predictionModelSpace"
        ]
    )

    native_reconstruction_passed = bool(
        predicted_native.shape
        == image_volume_native.shape
        and predicted_native_sitk.GetSize()
        == image_sitk.GetSize()
    )
else:
    predicted_native = None
    predicted_native_sitk = None
    native_reconstruction_passed = False

## 17. Metricas por clase, slice y volumen


In [ ]:
def class_metrics_from_arrays(gt: np.ndarray, pred: np.ndarray) -> list[dict[str, Any]]:
    rows = []
    for class_id, class_name in enumerate(CLASS_NAMES):
        gt_c = gt == class_id
        pred_c = pred == class_id
        tp = int(np.logical_and(gt_c, pred_c).sum())
        gt_count = int(gt_c.sum())
        pred_count = int(pred_c.sum())
        fp = pred_count - tp
        fn = gt_count - tp
        union = int(np.logical_or(gt_c, pred_c).sum())
        dice_den = gt_count + pred_count
        iou_den = union
        rows.append({
            "classId": class_id,
            "className": class_name,
            "predictionVoxels": pred_count,
            "groundTruthVoxels": gt_count,
            "intersection": tp,
            "union": union,
            "dice": None if dice_den == 0 else float(2 * tp / dice_den),
            "iou": None if iou_den == 0 else float(tp / iou_den),
            "presentInPrediction": bool(pred_count > 0),
            "presentInGroundTruth": bool(gt_count > 0),
            "bothEmpty": bool(dice_den == 0),
        })
    return rows


def macro_no_background(rows: list[dict[str, Any]], metric: str) -> float | None:
    values = [float(row[metric]) for row in rows if int(row["classId"]) > 0 and row[metric] is not None]
    return float(np.mean(values)) if values else None


def slice_metrics(gt_model: np.ndarray, pred_model: np.ndarray, conf_model: np.ndarray) -> pd.DataFrame:
    rows = []
    for idx in range(gt_model.shape[0]):
        class_rows = class_metrics_from_arrays(gt_model[idx], pred_model[idx])
        gt_fg = gt_model[idx] > 0
        pred_fg = pred_model[idx] > 0
        fp = np.logical_and(pred_fg, ~gt_fg)
        rows.append({
            "sliceIndex": idx,
            "hasGroundTruthForeground": bool(gt_fg.any()),
            "hasPredictedForeground": bool(pred_fg.any()),
            "foregroundVoxels": int(gt_fg.sum()),
            "diceMacroNoBackground": macro_no_background(class_rows, "dice"),
            "iouMacroNoBackground": macro_no_background(class_rows, "iou"),
            "falsePositiveRatio": float(fp.sum() / max(pred_fg.sum(), 1)),
            "meanConfidence": float(conf_model[idx].mean()),
            "meanForegroundConfidence": float(conf_model[idx][pred_fg].mean()) if pred_fg.any() else None,
        })
    return pd.DataFrame(rows)


def volume_metrics(gt_model: np.ndarray, pred_model: np.ndarray, conf_model: np.ndarray) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, dict[str, Any]]:
    class_rows = class_metrics_from_arrays(gt_model, pred_model)
    by_class = pd.DataFrame(class_rows)
    by_slice = slice_metrics(gt_model, pred_model, conf_model)
    fg_gt = gt_model > 0
    fg_pred = pred_model > 0
    fg_intersection = int(np.logical_and(fg_gt, fg_pred).sum())
    fg_union = int(np.logical_or(fg_gt, fg_pred).sum())
    fg_den = int(fg_gt.sum() + fg_pred.sum())
    case_row = {
        "caseAlias": case_alias,
        "diceMacroNoBackground": macro_no_background(class_rows, "dice"),
        "iouMacroNoBackground": macro_no_background(class_rows, "iou"),
        "diceMicroForeground": None if fg_den == 0 else float(2 * fg_intersection / fg_den),
        "iouMicroForeground": None if fg_union == 0 else float(fg_intersection / fg_union),
        "foregroundVoxelRatio": float(fg_gt.mean()),
        "presentClasses": [CLASS_NAMES[i] for i in CLASS_IDS if np.any(gt_model == i) or np.any(pred_model == i)],
        "informativeSlices": int(by_slice["hasGroundTruthForeground"].sum()),
        "emptySlices": int((~by_slice["hasGroundTruthForeground"]).sum()),
    }
    by_case = pd.DataFrame([case_row])
    metrics_json = {
        "case": case_row,
        "perClass": class_rows,
        "globalReference": {
            "testDiceMacroNoBackground": 0.8934316063846954,
            "testIouMacroNoBackground": 0.8079981902423006,
            "warning": "Resultado de un caso individual del split de test. No reemplaza la evaluacion agregada de los 33 pacientes y no constituye validacion clinica.",
        },
    }
    return by_class, by_slice, by_case, metrics_json


if run_gate_enabled:
    metrics_by_class_df, metrics_by_slice_df, metrics_by_case_df, full_volume_metrics = volume_metrics(
        volume_prediction["groundTruthModelSpace"],
        volume_prediction["predictionModelSpace"],
        volume_prediction["confidenceModelSpace"],
    )
else:
    metrics_by_class_df = pd.DataFrame()
    metrics_by_slice_df = pd.DataFrame()
    metrics_by_case_df = pd.DataFrame()
    full_volume_metrics = {}

## 18. Visualizaciones


In [ ]:
def choose_visual_slice_indices(metrics_df: pd.DataFrame) -> list[int]:
    if metrics_df.empty:
        return []
    fg = metrics_df.loc[metrics_df["hasGroundTruthForeground"]].copy()
    chosen: list[int] = []
    if not fg.empty:
        ordered = fg.sort_values("diceMacroNoBackground", na_position="last")
        chosen.extend([int(ordered.iloc[-1]["sliceIndex"]), int(ordered.iloc[len(ordered) // 2]["sliceIndex"]), int(ordered.iloc[0]["sliceIndex"])])
        chosen.append(int(fg.sort_values("foregroundVoxels").iloc[-1]["sliceIndex"]))
    empty = metrics_df.loc[~metrics_df["hasGroundTruthForeground"]].copy()
    if not empty.empty:
        chosen.append(int(empty.sort_values("falsePositiveRatio").iloc[-1]["sliceIndex"]))
    deduped = []
    for idx in chosen:
        if idx not in deduped:
            deduped.append(idx)
    return deduped


def save_visualizations(run_dir: Path) -> tuple[Path, Path, Path | None]:
    chosen = choose_visual_slice_indices(metrics_by_slice_df)
    preview_path = run_dir / "full_volume_best_median_worst_preview.png"
    profile_path = run_dir / "slice_dice_profile.png"
    diff_placeholder = run_dir / "runtime_direct_difference.png"
    if not chosen:
        return preview_path, profile_path, diff_placeholder
    fig, axes = plt.subplots(len(chosen), 6, figsize=(18, 3 * len(chosen)))
    if len(chosen) == 1:
        axes = np.expand_dims(axes, 0)
    for row, idx in enumerate(chosen):
        image_model, gt_model = preprocess_pair(idx)
        pred = volume_prediction["predictionModelSpace"][idx]
        tp = np.logical_and(gt_model > 0, pred > 0)
        fp = np.logical_and(gt_model == 0, pred > 0)
        fn = np.logical_and(gt_model > 0, pred == 0)
        error = np.zeros((*gt_model.shape, 3), dtype=np.float32)
        error[tp] = (0.0, 0.8, 0.0)
        error[fp] = (1.0, 0.2, 0.0)
        error[fn] = (0.2, 0.2, 1.0)
        panels = [image_model, gt_model, pred, gt_model, pred, error]
        titles = ["RM", "GT", "Pred", "Overlay GT", "Overlay pred", "Error"]
        for col, panel in enumerate(panels):
            axes[row, col].imshow(image_model, cmap="gray")
            if col in {1, 2, 3, 4}:
                axes[row, col].imshow(np.ma.masked_where(panel == 0, panel), alpha=0.55, vmin=0, vmax=3)
            elif col == 5:
                axes[row, col].imshow(error, alpha=0.75)
            axes[row, col].set_title(f"{titles[col]} slice={idx}")
            axes[row, col].axis("off")
    fig.tight_layout()
    fig.savefig(preview_path, dpi=140)
    plt.close(fig)

    fig, ax1 = plt.subplots(figsize=(12, 5))
    ax1.plot(metrics_by_slice_df["sliceIndex"], metrics_by_slice_df["diceMacroNoBackground"], label="Dice")
    ax1.plot(metrics_by_slice_df["sliceIndex"], metrics_by_slice_df["iouMacroNoBackground"], label="IoU")
    ax1.axvline(integration_slice_index, color="black", linestyle="--", label="runtime slice")
    ax2 = ax1.twinx()
    ax2.plot(metrics_by_slice_df["sliceIndex"], metrics_by_slice_df["foregroundVoxels"], color="gray", alpha=0.4, label="GT fg")
    ax1.legend(loc="upper left")
    ax2.legend(loc="upper right")
    fig.tight_layout()
    fig.savefig(profile_path, dpi=140)
    plt.close(fig)
    return preview_path, profile_path, diff_placeholder

## 19. Integracion con el runtime existente


In [ ]:
def sanitize_response(value: Any) -> Any:
    if isinstance(value, dict):
        return {
            key: sanitize_response(item)
            for key, item in value.items()
            if key not in {
                "input_path",
                "inputPath",
                "sourcePath",
                "outputFiles",
                "path",
            }
        }

    if isinstance(value, list):
        return [
            sanitize_response(item)
            for item in value
        ]

    return value


def prepare_runtime_canonical_input() -> Path:
    runtime_input_path = (
        CFG.local_work_dir
        / "runtime_input_canonical.npy"
    )

    temporary_path = (
        CFG.local_work_dir
        / "runtime_input_canonical.tmp.npy"
    )

    np.save(
        temporary_path,
        np.asarray(
            image_volume,
            dtype=np.float32,
        ),
        allow_pickle=False,
    )

    os.replace(
        temporary_path,
        runtime_input_path,
    )

    verified = np.load(
        runtime_input_path,
        allow_pickle=False,
    )

    if verified.shape != image_volume.shape:
        raise RuntimeError(
            "El volumen canonicalizado para runtime "
            "tiene shape incorrecta."
        )

    if not np.isfinite(verified).all():
        raise RuntimeError(
            "El volumen canonicalizado para runtime "
            "contiene valores no finitos."
        )

    return runtime_input_path


def run_runtime_integration(
    run_dir: Path,
) -> tuple[
    dict[str, Any],
    np.ndarray,
    np.ndarray,
]:
    runtime_model_dir = (
        CFG.local_work_dir
        / "runtime_models"
    )

    runtime_output_dir = (
        CFG.local_work_dir
        / "runtime_outputs"
    )

    runtime_model_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    runtime_output_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    for filename in [
        (
            "sagittal_spider_"
            "multiclass_final_best.pt"
        ),
        (
            "sagittal_spider_"
            "multiclass_final_best.pt."
            "manifest.json"
        ),
    ]:
        source = release_paths[filename]
        destination = (
            runtime_model_dir
            / filename
        )

        shutil.copy2(
            source,
            destination,
        )

        if (
            sha256_stream(destination)
            != sha256_stream(source)
        ):
            raise RuntimeError(
                f"Copia runtime corrupta: "
                f"{filename}"
            )

    runtime_input_path = (
        prepare_runtime_canonical_input()
    )

    os.environ["PFI_MODEL_DIR"] = str(
        runtime_model_dir
    )

    os.environ["PFI_OUTPUT_DIR"] = str(
        runtime_output_dir
    )

    os.environ["PFI_INFERENCE_DEVICE"] = (
        CFG.device
    )

    from pfi_ai_service.real_inference_runtime import (
        clear_model_cache,
    )
    from pfi_ai_service.pipeline import (
        PipelineRunRequest,
        run_pipeline,
    )

    clear_model_cache()

    trace_id = (
        "trace-"
        + short_hash(
            case_alias
            + CFG.expected_release_content_sha256,
            12,
        )
    )

    request = PipelineRunRequest(
        caseId=case_alias,
        plane="sagittal",
        modelKey=CFG.model_key,
        inputPath=str(runtime_input_path),
        metadata={
            "inferenceMode": "real_baseline",
            "allowContractFallback": False,
            "traceId": trace_id,
            "patientId": case_alias,
            "sliceIndex": int(
                integration_slice_index
            ),
            "sliceAxis": int(
                CFG.expected_sagittal_axis
            ),
            "deidentified": True,
            "inputOrientation": (
                "spider_canonical_y_x_slice"
            ),
        },
    )

    response = run_pipeline(request)

    ai_output = response.get(
        "aiOutput",
        {},
    )

    metadata = response.get(
        "metadata",
        {},
    )

    if (
        ai_output.get("inferenceMode")
        != "real_baseline"
        or metadata.get("inferenceMode")
        != "real_baseline"
    ):
        raise RuntimeError(
            "Runtime no quedó en real_baseline."
        )

    if (
        metadata.get(
            "requestedInferenceMode"
        )
        != "real_baseline"
    ):
        raise RuntimeError(
            "requestedInferenceMode inesperado."
        )

    if (
        response.get("modelKey")
        != CFG.model_key
        or response.get("modelVersion")
        != CFG.version
    ):
        raise RuntimeError(
            "Modelo o versión runtime inesperados."
        )

    if (
        metadata.get("artifactHash")
        != CFG.expected_model_sha256
    ):
        raise RuntimeError(
            "artifactHash runtime inesperado."
        )

    if (
        int(metadata.get("selectedSlice"))
        != int(integration_slice_index)
    ):
        raise RuntimeError(
            "Slice runtime inesperada."
        )

    if (
        int(metadata.get("selectedAxis"))
        != CFG.expected_sagittal_axis
    ):
        raise RuntimeError(
            "Eje runtime inesperado."
        )

    if (
        int(metadata.get("sliceCount"))
        != int(image_slice_count)
    ):
        raise RuntimeError(
            "sliceCount runtime inesperado."
        )

    if (
        metadata.get(
            "allowContractFallback"
        )
        is not False
        or "realInferenceFailure"
        in metadata
    ):
        raise RuntimeError(
            "Fallback runtime detectado."
        )

    output_files = metadata.get(
        "outputFiles",
        {},
    )

    mask_path = Path(
        output_files["maskPath"]
    )

    confidence_path = Path(
        output_files["confidencePath"]
    )

    overlay_path = Path(
        output_files["overlayPath"]
    )

    for path in [
        mask_path,
        confidence_path,
        overlay_path,
    ]:
        if not path.exists():
            raise RuntimeError(
                "Output runtime faltante: "
                f"{path.name}"
            )

    runtime_mask = np.load(
        mask_path,
        allow_pickle=False,
    )

    runtime_confidence = np.load(
        confidence_path,
        allow_pickle=False,
    )

    if (
        runtime_mask.shape
        != CFG.expected_target_size
    ):
        raise RuntimeError(
            "Mask runtime con shape inválida."
        )

    runtime_labels = {
        int(value)
        for value in np.unique(
            runtime_mask
        )
    }

    if not runtime_labels.issubset(
        set(CLASS_IDS)
    ):
        raise RuntimeError(
            "Mask runtime con labels inválidos."
        )

    if not np.isfinite(
        runtime_confidence
    ).all():
        raise RuntimeError(
            "Confidence runtime no finita."
        )

    sanitized = sanitize_response(
        response
    )

    atomic_write_json(
        run_dir
        / "runtime_integration_response.json",
        sanitized,
    )

    return (
        sanitized,
        runtime_mask.astype(np.uint8),
        runtime_confidence.astype(
            np.float32
        ),
    )

## 20. Comparacion runtime contra inferencia directa


In [ ]:
def compare_runtime_direct(runtime_mask: np.ndarray, run_dir: Path) -> dict[str, Any]:
    direct_mask = volume_prediction["predictionModelSpace"][int(integration_slice_index)]
    if runtime_mask.shape != direct_mask.shape:
        raise RuntimeError("Shape runtime/direct distinto.")
    different = runtime_mask != direct_mask
    class_rows = class_metrics_from_arrays(direct_mask, runtime_mask)
    pixel_agreement = float((~different).mean())
    diff_path = run_dir / "runtime_direct_difference.png"
    Image.fromarray((different.astype(np.uint8) * 255)).save(diff_path)
    result = {
        "runtimeEquivalencePassed": bool(pixel_agreement == 1.0 and int(different.sum()) == 0),
        "pixelAgreement": pixel_agreement,
        "differentPixels": int(different.sum()),
        "sliceIndex": int(integration_slice_index),
        "diceByClass": class_rows,
        "differenceImage": diff_path.name,
    }
    atomic_write_json(run_dir / "runtime_direct_equivalence.json", result)
    if not result["runtimeEquivalencePassed"]:
        raise RuntimeError("Runtime y evaluacion directa no son equivalentes.")
    return result

## 21. Persistencia atomica de evidencia en Drive


In [ ]:
def atomic_write_text(path: Path, text: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    fd, tmp_name = tempfile.mkstemp(prefix=path.name, suffix=".tmp", dir=str(path.parent))
    with os.fdopen(fd, "w", encoding="utf-8") as fh:
        fh.write(text)
        fh.flush()
        os.fsync(fh.fileno())
    os.replace(tmp_name, path)


def atomic_write_json(path: Path, payload: dict[str, Any]) -> None:
    atomic_write_text(path, stable_json_dumps(payload) + "\n")


def atomic_write_csv(path: Path, frame: pd.DataFrame) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    frame.to_csv(tmp, index=False, encoding="utf-8")
    os.replace(tmp, path)


def artifact_entry(path: Path, required: bool = True) -> dict[str, Any]:
    return {
        "filename": path.name,
        "sizeBytes": int(path.stat().st_size),
        "sha256": sha256_stream(path),
        "contentType": "application/json" if path.suffix == ".json" else "text/csv" if path.suffix == ".csv" else "image/png" if path.suffix == ".png" else "application/octet-stream",
        "required": required,
    }


def run_id_for_case() -> str:
    raw = CFG.expected_release_content_sha256 + patient_hash + image_object + CFG.expected_model_sha256
    return hashlib.sha256(raw.encode("utf-8")).hexdigest()[:16]


run_id = run_id_for_case()
run_dir = CFG.drive_output_root / "runs" / run_id
if run_gate_enabled:
    run_dir.mkdir(parents=True, exist_ok=True)
    existing_manifest = run_dir / "full_volume_evidence_manifest.json"
    if existing_manifest.exists():
        existing = read_json(existing_manifest)
        if existing.get("releaseContentSha256") != CFG.expected_release_content_sha256 or existing.get("patientIdHash") != patient_hash or existing.get("modelSha256") != CFG.expected_model_sha256:
            raise RuntimeError("Run existente incompatible.")
    atomic_write_json(run_dir / "release_verification.json", release_verification)
    atomic_write_json(run_dir / "split_verification.json", split_verification)
    atomic_write_json(run_dir / "selected_case_manifest.json", selected_case_manifest)
    atomic_write_json(run_dir / "preprocessing_validation.json", preprocessing_validation)
    atomic_write_json(run_dir / "full_volume_metrics.json", full_volume_metrics)
    atomic_write_csv(run_dir / "full_volume_metrics_by_class.csv", metrics_by_class_df)
    atomic_write_csv(run_dir / "full_volume_metrics_by_slice.csv", metrics_by_slice_df)
    atomic_write_csv(run_dir / "full_volume_metrics_by_case.csv", metrics_by_case_df)
    sitk.WriteImage(predicted_native_sitk, str(run_dir / "predicted_segmentation_native.mha"))
    if volume_prediction["predictionModelSpace"].nbytes < 256 * 1024 * 1024:
        tmp_npy = run_dir / "predicted_segmentation_model_space.tmp.npy"
        np.save(tmp_npy, volume_prediction["predictionModelSpace"])
        os.replace(tmp_npy, run_dir / "predicted_segmentation_model_space.npy")
    preview_path, profile_path, _ = save_visualizations(run_dir)
    runtime_response, runtime_mask, runtime_confidence = run_runtime_integration(run_dir)
    runtime_equivalence = compare_runtime_direct(runtime_mask, run_dir)
else:
    runtime_response = {}
    runtime_equivalence = {}

## 22. Manifest de evidencia y hashes


In [ ]:
def deterministic_evidence_content(payload: dict[str, Any]) -> str:
    stable = {
        key: value
        for key, value in payload.items()
        if key not in {"createdAtUtc", "evidenceContentSha256"}
    }
    return hashlib.sha256(json.dumps(stable, sort_keys=True, separators=(",", ":"), ensure_ascii=False).encode("utf-8")).hexdigest()


if run_gate_enabled:
    evidence_files = [
        "release_verification.json",
        "split_verification.json",
        "selected_case_manifest.json",
        "preprocessing_validation.json",
        "full_volume_metrics.json",
        "full_volume_metrics_by_class.csv",
        "full_volume_metrics_by_slice.csv",
        "full_volume_metrics_by_case.csv",
        "predicted_segmentation_native.mha",
        "full_volume_best_median_worst_preview.png",
        "slice_dice_profile.png",
        "runtime_integration_response.json",
        "runtime_direct_equivalence.json",
        "runtime_direct_difference.png",
    ]
    model_space_path = run_dir / "predicted_segmentation_model_space.npy"
    if model_space_path.exists():
        evidence_files.append(model_space_path.name)
    artifacts = [artifact_entry(run_dir / name) for name in evidence_files if (run_dir / name).exists()]
    source_direction_hash = hashlib.sha256(json.dumps(geometry_payload(image_sitk)["direction"], sort_keys=True).encode("utf-8")).hexdigest()
    evidence_manifest = {
        "schemaVersion": 1,
        "runId": run_id,
        "caseAlias": case_alias,
        "patientIdHash": patient_hash,
        "releaseId": CFG.release_id,
        "releaseContentSha256": CFG.expected_release_content_sha256,
        "releaseManifestSha256": CFG.expected_release_manifest_sha256,
        "modelSha256": CFG.expected_model_sha256,
        "datasetManifestSha256": CFG.expected_dataset_manifest_sha256,
        "trainingIndexSha256": CFG.expected_training_index_sha256,
        "sourceTrainingCommit": CFG.expected_training_commit,
        "publisherCommit": PFI_RELEASE_REPO_REF,
        "evaluationNotebook": "notebooks/47_gcs_sagittal_final_full_volume_inference.ipynb",
        "selectionMode": selection_mode,
        "split": "test",
        "imageObject": image_object,
        "maskObject": mask_object,
        "sourceShape": list(image_volume_native.shape),"canonicalShape": list(image_volume.shape),
        "sourceSpacing": geometry_payload(image_sitk)["spacing"],
        "sourceDirectionHash": source_direction_hash,
        "sagittalAxis": CFG.expected_sagittal_axis,
        "sliceCount": int(image_volume.shape[CFG.expected_sagittal_axis]),
        "targetSize": list(CFG.expected_target_size),
        "metrics": full_volume_metrics.get("case", {}),
        "runtimeIntegration": {"passed": bool(runtime_response)},
        "runtimeEquivalence": runtime_equivalence,
        "humanReviewRequired": CFG.human_review_required,
        "notClinicalDiagnosis": CFG.not_clinical_diagnosis,
        "clinicalValidationStatus": "not_clinically_validated",
        "artifacts": artifacts,
        "totalFiles": len(artifacts),
        "totalBytes": int(sum(item["sizeBytes"] for item in artifacts)),
        "createdAtUtc": datetime.now(timezone.utc).isoformat(),
    }
    evidence_manifest["evidenceContentSha256"] = deterministic_evidence_content(evidence_manifest)
    atomic_write_json(run_dir / "full_volume_evidence_manifest.json", evidence_manifest)
    success_payload = {
        "runId": run_id,
        "status": "full_volume_evaluation_completed",
        "releaseVerified": True,
        "splitVerified": True,
        "selectedCaseIsTest": True,
        "strictLoadSuccess": bool(strict_load_success),
        "allSlicesProcessed": True,
        "nativeReconstructionPassed": bool(native_reconstruction_passed),
        "metricsGenerated": True,
        "runtimeRealBaselinePassed": bool(runtime_response),
        "runtimeFallbackUsed": False,
        "runtimeEquivalencePassed": bool(runtime_equivalence.get("runtimeEquivalencePassed")),
        "gcsReadOnly": gcs_write_operations == 0,
        "gcsWriteOperations": gcs_write_operations,
        "humanReviewRequired": CFG.human_review_required,
        "notClinicalDiagnosis": CFG.not_clinical_diagnosis,
        "clinicalValidationStatus": "not_clinically_validated",
        "evidenceContentSha256": evidence_manifest["evidenceContentSha256"],
        "createdAtUtc": datetime.now(timezone.utc).isoformat(),
    }
    atomic_write_json(run_dir / "FULL_VOLUME_EVALUATION_SUCCESS.json", success_payload)
else:
    evidence_manifest = {}
    success_payload = {}

## 23. Resumen final estricto


In [ ]:
if run_gate_enabled:
    final_summary = {
        "run_id": run_id,
        "execution_status": "full_volume_evaluation_completed",
        "release_verified": True,
        "release_content_sha256": CFG.expected_release_content_sha256,
        "release_manifest_sha256": CFG.expected_release_manifest_sha256,
        "model_sha256": CFG.expected_model_sha256,
        "split_verified": True,
        "selection_mode": selection_mode,
        "case_alias": case_alias,
        "selected_case_is_test": True,
        "patient_leakage_detected": False,
        "image_shape": list(image_volume_native.shape),
        "mask_shape": list(grouped_mask_volume_native.shape),
        "canonical_image_shape": list(image_volume.shape),
        "sagittal_axis": CFG.expected_sagittal_axis,
        "slices_processed": int(volume_prediction["stats"]["totalSlices"]),
        "classes": CLASS_NAMES,
        "volume_dice_macro_no_bg": full_volume_metrics["case"]["diceMacroNoBackground"],
        "volume_iou_macro_no_bg": full_volume_metrics["case"]["iouMacroNoBackground"],
        "per_class_metrics": full_volume_metrics["perClass"],
        "predicted_native_shape": list(predicted_native.shape),
        "native_geometry_preserved": bool(native_reconstruction_passed),
        "runtime_real_baseline_passed": bool(runtime_response),
        "runtime_selected_slice": int(integration_slice_index),
        "runtime_fallback_used": False,
        "runtime_pixel_agreement": runtime_equivalence.get("pixelAgreement"),
        "runtime_different_pixels": runtime_equivalence.get("differentPixels"),
        "runtime_equivalence_passed": runtime_equivalence.get("runtimeEquivalencePassed"),
        "evidence_dir": str(run_dir),
        "evidence_manifest_sha256": sha256_stream(run_dir / "full_volume_evidence_manifest.json"),
        "gcs_read_operations": gcs_read_operations,
        "gcs_write_operations": gcs_write_operations,
        "full_volume_inference_success": True,
    }
else:
    final_summary = {
        "run_id": run_id,
        "execution_status": "preflight_ready",
        "release_verified": True,
        "release_content_sha256": CFG.expected_release_content_sha256,
        "release_manifest_sha256": CFG.expected_release_manifest_sha256,
        "model_sha256": CFG.expected_model_sha256,
        "split_verified": True,
        "selection_mode": selection_mode,
        "case_alias": case_alias,
        "selected_case_is_test": True,
        "patient_leakage_detected": False,
        "image_shape": list(image_volume.shape),
        "mask_shape": list(raw_mask_volume.shape),
        "sagittal_axis": CFG.expected_sagittal_axis,
        "slices_processed": 0,
        "classes": CLASS_NAMES,
        "volume_dice_macro_no_bg": None,
        "volume_iou_macro_no_bg": None,
        "per_class_metrics": [],
        "predicted_native_shape": None,
        "native_geometry_preserved": False,
        "runtime_real_baseline_passed": False,
        "runtime_selected_slice": int(integration_slice_index),
        "runtime_fallback_used": False,
        "runtime_pixel_agreement": None,
        "runtime_different_pixels": None,
        "runtime_equivalence_passed": False,
        "evidence_dir": str(CFG.drive_output_root / "runs" / run_id),
        "evidence_manifest_sha256": None,
        "gcs_read_operations": gcs_read_operations,
        "gcs_write_operations": gcs_write_operations,
        "full_volume_inference_success": False,
    }
if gcs_write_operations != 0:
    raise RuntimeError("GCS write operations debe permanecer en cero.")
print(stable_json_dumps(final_summary))